In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten

In [3]:
df=pd.read_csv('training.csv')
df.head(10)

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3
5,ive been feeling a little burdened lately wasn...,0
6,ive been taking or milligrams or times recomme...,5
7,i feel as confused about life as a teenager or...,4
8,i have been with petronas for years i feel tha...,1
9,i feel romantic too,2


In [4]:
df.tail()

,text,label
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,1
15998,i feel like this was such a rude comment and i...,3
15999,i know a lot but i feel so stupid because i ca...,0


In [5]:
df.describe()

,label
count,16000.000000
mean,1.565937
std,1.501430
min,0.000000
25%,0.000000
50%,1.000000
75%,3.000000
max,5.000000


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    16000 non-null  object
 1   label   16000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 250.1+ KB


In [7]:
df.shape

(16000, 2)

In [8]:
df.fillna(0,inplace=True)

In [9]:
df['text'][1]

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [10]:
import re
print(df.iloc[2].text)

im grabbing a minute to post i feel greedy wrong


In [11]:
clean=re.compile('<.*?>')
print(re.sub(clean,'',df.iloc[1].text))

i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake


In [12]:
def clean_html(text):
  clean=re.compile('<.*?>')
  return re.sub(clean,'',text)

In [13]:
df['text']=df['text'].apply(clean_html)

In [14]:
df.head()

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


In [15]:
def remove_special(text):
  x=''
  for i in text:
    if i.isalnum():
      x=x+i
    else:
      x=x+' '
  return x

In [16]:
remove_special("Th%e @ classic use of the word. it is called oz as that is the nickname given to the oswald maximum")

'Th e   classic use of the word  it is called oz as that is the nickname given to the oswald maximum'

In [17]:
df['text']=df['text'].apply(remove_special)

In [18]:
df.head()

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


In [19]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [20]:
from nltk.corpus import stopwords
stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [21]:
def remove_stopwords(text):
  x=[]
  for i in text.split():
    if i not in stopwords.words('english'):
      x.append(i)
  y=x[:]
  x.clear()
  return y

In [22]:
df['text']=df['text'].apply(remove_stopwords)

In [23]:
print (df.head())

                                                text  label
0                          [didnt, feel, humiliated]      0
1  [go, feeling, hopeless, damned, hopeful, aroun...      0
2  [im, grabbing, minute, post, feel, greedy, wrong]      3
3  [ever, feeling, nostalgic, fireplace, know, st...      2
4                                 [feeling, grouchy]      3


In [24]:
y=[]
from nltk.stem import PorterStemmer
ps=PorterStemmer()
def stem_words(text):
  for i in text:
    y.append(ps.stem(i))
  z=y[:]
  y.clear()
  return z

In [25]:
df['text']=df['text'].apply(stem_words)

In [26]:
df.head()

,text,label
0,"[didnt, feel, humili]",0
1,"[go, feel, hopeless, damn, hope, around, someo...",0
2,"[im, grab, minut, post, feel, greedi, wrong]",3
3,"[ever, feel, nostalg, fireplac, know, still, p...",2
4,"[feel, grouchi]",3


In [27]:
def join_back(list_input):
  return " ".join(list_input)

In [28]:
df['text']=df['text'].apply(join_back)

In [29]:
df.head()

,text,label
0,didnt feel humili,0
1,go feel hopeless damn hope around someon care ...,0
2,im grab minut post feel greedi wrong,3
3,ever feel nostalg fireplac know still properti,2
4,feel grouchi,3


In [30]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=150)

In [31]:
X=cv.fit_transform(df['text']).toarray()

In [32]:
X[1:5,:]

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [33]:
X.shape

(16000, 150)

In [34]:
y=df.iloc[:,-1].values

In [35]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,y,test_size=0.2)

In [36]:
model=Sequential()

In [37]:
model.add(Conv1D(filters=32,kernel_size=2, activation='relu',input_shape=(X_train.shape[1],1)))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [38]:
model.add(MaxPooling1D(pool_size=2))

In [39]:
model.add(Flatten())

In [40]:
from tensorflow.keras.layers import Dense
model.add(Dense(16,activation='relu'))

In [41]:
model.add(Dense(6,activation='softmax'))

In [42]:
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [43]:
from tensorflow.keras.utils import to_categorical

# One-hot encode the target variable
Y_train_encoded = to_categorical(Y_train, num_classes=6)
Y_test_encoded = to_categorical(Y_test, num_classes=6)

model.fit(X_train, Y_train_encoded, epochs=800, batch_size=16)

Epoch 1/800
800/800 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.3563 - loss: 1.5536
Epoch 2/800
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.3791 - loss: 1.4939
Epoch 3/800
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.3898 - loss: 1.4738
Epoch 4/800
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.3916 - loss: 1.4636
Epoch 5/800
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.3995 - loss: 1.4541
Epoch 6/800
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4065 - loss: 1.4474
Epoch 7/800
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4061 - loss: 1.4389
Epoch 8/800
800/800 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.4119 - loss: 1.4321
Epoch 9/800
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.4109 - loss: 1.4274
Epoch 10/800
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4205 - loss: 1.4233
Epoch 11/800
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.4199 - loss: 1.4189
Epoch 12/800
800/800 ━━━━━━━━━━━━━━━━━━━━

In [44]:
loss,accuracy=model.evaluate(X_test,Y_test_encoded)
print("Accuracy",accuracy)

100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3541 - loss: 3.9512
Accuracy 0.3540624976158142
